# Side-View LPG Cylinder Classifier — EfficientNetB2 + Spatial Attention (v6/v7)

## Overview
Trains the primary side-view brand classifier used by the main inference paths
(`src/predict.py` and the side-view branch of `src/predict_ensemble.py`). Given a whole-bbox
crop of a cylinder from an angled kiosk-style photo, it predicts the brand. This notebook
adds a spatial-attention module on top of an EfficientNetB2 backbone so the model learns to
weight label/logo regions over background, and trains on a 3-class crop dataset (the
`unknown` class present in the raw dataset is deliberately dropped for this run — see cell
"Cell 2").

## How to Run
1. **Runtime:** Colab GPU (a T4 handled this run at ~20-23s/epoch for 50 epochs;
   CPU-only would be impractical).
2. **Upload/mount:** No Drive mount cell up front — Drive is only touched later, when saving
   results (`DRIVE_SAVE` path in "Cell 7"). Upload the dataset zip when prompted in
   "Cell 2 — Upload + Unzip Dataset" (interactive `files.upload()`; this run used
   `dataset_cropped_v3_clean.zip`).
3. **Execution order:** Run top to bottom. Note cell 4 (`shutil.rmtree(...unknown)`)
   permanently deletes the `unknown` class folders from the extracted dataset in the Colab
   VM (not the source zip) so training proceeds as 3-class — this is intentional for this
   run, not a bug. The GradCAM section near the end has two variants: a simple upload-and-run
   cell, and a later drag-and-drop widget version — either can be run standalone once a
   trained model is in memory.
4. **Expected outputs:** best checkpoint (saved on every val-acc improvement), training
   history JSON, confusion-matrix PNG, training-curve PNG, and ad-hoc GradCAM PNGs — see the
   Output Files section near the end of the notebook.

## Model / Dataset Info
| | |
|---|---|
| Architecture | EfficientNetB2 backbone + SpatialAttention head (`LPGClassifierAttention`) |
| Dataset | `dataset_cropped_v3_clean.zip`, 3-class after `unknown` removal — train 2545 / val 1093 (bharat 934/401, hp 732/314, indane 879/378 train/val) |
| Classes | bharat, hp, indane |
| Expected accuracy | Best val acc reached during this run: **96.3%** (epoch 38), logged in `history` / printed as "Done! Best val acc: 96.3%" |

## Current Status
Training, evaluation, and GradCAM-visualization code are all complete and were run to
completion (50/50 epochs) — this notebook's embedded outputs still show the full run.
Naming is inconsistent throughout: the notebook is headed "v6 Training" and several print
statements/plot titles still say "v6", but the checkpoint/history/plot filenames and the
Drive save comment say "v7" (`classifier_best_v7_attention_3_class.pth`,
`DRIVE_SAVE = ".../Classifier/v7_attention"`). Per project records this run corresponds to
the model documented elsewhere as **v6** (`classifier_best_v6_attention.pth`, val 95.15%) —
the 96.3% figure and "v7" filenames in this notebook are from this specific training
execution and are not reconciled with that naming. Left as-is since fixing would require
guessing which label is authoritative. The final two GradCAM cells are near-duplicates: the
first ("Cell 9" section, upload-based) is superseded by the second (drag-and-drop widget with
EXIF auto-orientation) but neither is an exact duplicate of the other, so both are left in
place — the last cell in the notebook is empty (no content).

## Cell 2 — Upload + Unzip Dataset

In [ ]:
from google.colab import files
import zipfile, os

print("Upload your dataset zip")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/dataset")
print("Unzipped!")

In [ ]:
# Check structure
for split in ["train", "valid"]:
    for brand in os.listdir(f"/content/dataset/dataset_cropped_v3_clean/{split}"):
        count = len(os.listdir(f"/content/dataset/dataset_cropped_v3_clean/{split}/{brand}"))
        print(f"{split}/{brand}: {count}")

In [ ]:
import shutil
shutil.rmtree("/content/dataset/dataset_cropped_v3_clean/train/unknown")
shutil.rmtree("/content/dataset/dataset_cropped_v3_clean/valid/unknown")
print("Unknown removed!")

## Cell 3 — Transforms

In [ ]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.ColorJitter(brightness=0.6, contrast=0.6, saturation=0.6, hue=0.2),
    transforms.RandomGrayscale(p=0.2),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5, scale=(0.05, 0.25), ratio=(0.3, 3.0)),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Transforms ready!")

## Cell 4 — Dataset + Sampler + Dataloaders

In [ ]:
import torch
from torchvision import datasets
from torch.utils.data import DataLoader, WeightedRandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

TRAIN_PATH = "/content/dataset/dataset_cropped_v3_clean/train"
VAL_PATH   = "/content/dataset/dataset_cropped_v3_clean/valid"

train_data = datasets.ImageFolder(TRAIN_PATH, transform=train_transforms)
val_data   = datasets.ImageFolder(VAL_PATH,   transform=val_transforms)

CLASSES   = train_data.classes
N_CLASSES = len(CLASSES)
print(f"Classes: {CLASSES}")
print(f"Train: {len(train_data)} | Val: {len(val_data)}")

# WeightedRandomSampler for class imbalance
class_counts   = [len([s for s in train_data.samples if s[1] == i]) for i in range(N_CLASSES)]
print(f"Class counts: {dict(zip(CLASSES, class_counts))}")
sample_weights = [1.0 / class_counts[s[1]] for s in train_data.samples]
sampler        = WeightedRandomSampler(sample_weights, len(train_data), replacement=True)

train_loader = DataLoader(train_data, batch_size=32, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_data,   batch_size=32, shuffle=False,   num_workers=2)

print("Dataloaders ready!")

## Cell 5 — Model: EfficientNetB2 + Spatial Attention

In [ ]:
import torch.nn as nn
from torchvision import models
class SpatialAttention(nn.Module):
    """Learns which spatial regions matter most for brand identification.
    Combines avg and max pooling across channels to produce an attention map,
    then multiplies back onto feature maps to highlight important regions
    (label band, logo area) and suppress background.
    """
    def __init__(self):
        super().__init__()
        self.conv    = nn.Conv2d(1, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_pool = x.mean(dim=1, keepdim=True)          # (B, 1, H, W)
        max_pool = x.max(dim=1, keepdim=True).values    # (B, 1, H, W)
        pooled   = avg_pool + max_pool                  # combined spatial signal
        att_map  = self.sigmoid(self.conv(pooled))      # (B, 1, H, W) — 0 to 1
        return x * att_map                              # highlight important regions


class LPGClassifierAttention(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        backbone      = models.efficientnet_b2(weights="IMAGENET1K_V1")
        self.features = backbone.features
        self.avgpool  = backbone.avgpool
        feat_dim      = backbone.classifier[1].in_features  # 1408 for B2

        self.attention = SpatialAttention()

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        feat_maps = self.features(x)          # (B, C, H, W)
        attended  = self.attention(feat_maps) # (B, C, H, W) — weighted
        pooled    = self.avgpool(attended)    # (B, C, 1, 1)
        flat      = torch.flatten(pooled, 1) # (B, feat_dim)
        return self.classifier(flat)


model     = LPGClassifierAttention(num_classes=N_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready!")
print(f"Architecture: EfficientNetB2 + Spatial Attention")
print(f"Trainable params: {total_params:,}")

## Cell 6 — Training Loop

In [ ]:
from tqdm import tqdm
import json
import numpy as np

EPOCHS       = 50
best_val_acc = 0.0
history      = []

for epoch in range(EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total   += labels.size(0)

    # ── Val ────────────────────────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs  = model(imgs)
            loss     = criterion(outputs, labels)
            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)

    train_acc      = train_correct / train_total * 100
    val_acc        = val_correct   / val_total   * 100
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss   = val_loss   / len(val_loader)

    history.append({
        "epoch":      epoch + 1,
        "train_loss": round(avg_train_loss, 4),
        "train_acc":  round(train_acc, 2),
        "val_loss":   round(avg_val_loss, 4),
        "val_acc":    round(val_acc, 2),
    })

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.1f}% | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.1f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict":     model.state_dict(),
            "best_val_acc":         best_val_acc,
            "classes":              CLASSES,
            "architecture":         "efficientnet_b2 + spatial_attention",
            "num_classes":          N_CLASSES,
            "epochs_trained":       epoch + 1,
            "train_history":        history,
            "class_counts":         dict(zip(CLASSES, class_counts)),
            "confidence_threshold": 0.60,
        }, "classifier_best_v7_attention_3_class.pth")
        print(f"  ✅ New best saved: {val_acc:.1f}%")

    scheduler.step()

print(f"\nDone! Best val acc: {best_val_acc:.1f}%")

with open("training_history_v7_attention_3_class.json", "w") as f:
    json.dump(history, f, indent=2)
print("Training history saved!")

## Cell 7 — Save to Drive + Download

In [ ]:
import shutil, os
from google.colab import files

DRIVE_SAVE = "/content/drive/MyDrive/LPG Cylinder Detection and Classification/Classifier/v7_attention"  # ← UPDATE THIS PATH
os.makedirs(DRIVE_SAVE, exist_ok=True)

# Note: source is v5_attention.pth (training loop name), saving as v6 to Drive
shutil.copy("classifier_best_v7_attention_3_class.pth",   f"{DRIVE_SAVE}/classifier_best_v7_attention_3_class.pth")
shutil.copy("training_history_v7_attention_3_class.json",  f"{DRIVE_SAVE}/training_history_v7_attention_3_class.json")
print("Saved to Drive!")

files.download("classifier_best_v7_attention_3_class.pth")

## Cell 8 — Evaluation: Classification Report + Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load best
checkpoint = torch.load("classifier_best_v7_attention_3_class.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded v6 attention — best val acc: {checkpoint['best_val_acc']:.1f}%")

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs    = imgs.to(device)
        outputs = model(imgs)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print("\n" + "="*55)
print("CLASSIFICATION REPORT — v7 Attention")
print("="*55)
print(classification_report(all_labels, all_preds, target_names=CLASSES))

# Confusion matrix
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm,      annot=True, fmt="d",   cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
sns.heatmap(cm_norm, annot=True, fmt=".0%", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[0].set_title("Raw Counts",      fontweight="bold")
axes[1].set_title("Normalised (%)",  fontweight="bold")
for ax in axes:
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")
plt.suptitle("EfficientNetB2 + Spatial Attention — v7", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix_v7_attention.png", dpi=150, bbox_inches="tight")
plt.show()

shutil.copy("confusion_matrix_v7_attention.png", f"{DRIVE_SAVE}/confusion_matrix_v7_attention.png")
files.download("confusion_matrix_v7_attention.png")
print("Done!")

## Cell 9 — Training History Plot

In [ ]:
import json

with open("training_history_v7_attention_3_class.json") as f:
    history = json.load(f)

epochs     = [h["epoch"]      for h in history]
train_acc  = [h["train_acc"]  for h in history]
val_acc    = [h["val_acc"]    for h in history]
train_loss = [h["train_loss"] for h in history]
val_loss   = [h["val_loss"]   for h in history]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(epochs, train_acc,  label="Train Acc",  color="#3498db", linewidth=2)
axes[0].plot(epochs, val_acc,    label="Val Acc",    color="#2ecc71", linewidth=2)
axes[0].axhline(y=max(val_acc), color="black", linestyle="--", linewidth=1,
                label=f"Best Val: {max(val_acc):.1f}%")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Accuracy over Epochs", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].plot(epochs, train_loss, label="Train Loss", color="#e74c3c", linewidth=2)
axes[1].plot(epochs, val_loss,   label="Val Loss",   color="#e67e22", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Loss over Epochs", fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].spines[["top", "right"]].set_visible(False)

plt.suptitle("EfficientNetB2 + Spatial Attention v6 — Training History",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("training_history_v7_attention_3_class.png", dpi=150, bbox_inches="tight")
plt.show()

shutil.copy("training_history_v7_attention_3_class.png", f"{DRIVE_SAVE}/training_history_v6_attention.png")
files.download("training_history_v7_attention_3_class.png")
print("Done!")

## Test images

In [ ]:
!pip install grad-cam -q
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from google.colab import files

print("Upload 4-5 test images")
uploaded_imgs = files.upload()
image_paths   = list(uploaded_imgs.keys())

transform_test = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Target last conv block — attention model uses self.features
target_layer = [model.features[-1]]

n   = len(image_paths)
fig, axes = plt.subplots(n, 3, figsize=(15, 5*n))
if n == 1: axes = [axes]

for i, path in enumerate(image_paths):
    img_pil = Image.open(path).convert("RGB").resize((224, 224))
    img_np  = np.array(img_pil).astype(np.float32) / 255.0
    tensor  = transform_test(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, dim=1)[0]

    pred_idx   = probs.argmax().item()
    pred_brand = CLASSES[pred_idx]
    confidence = round(probs[pred_idx].item() * 100, 1)

    with GradCAM(model=model, target_layers=target_layer) as cam:
        grayscale = cam(input_tensor=tensor,
                       targets=[ClassifierOutputTarget(pred_idx)])
        cam_image = show_cam_on_image(img_np, grayscale[0], use_rgb=True)

    axes[i][0].imshow(img_np)
    axes[i][0].set_title(f"Original\n{path}", fontsize=10)
    axes[i][0].axis("off")

    axes[i][1].imshow(cam_image)
    axes[i][1].set_title(f"GradCAM\n{pred_brand} ({confidence}%)", fontsize=10)
    axes[i][1].axis("off")

    probs_list = [round(probs[j].item()*100, 1) for j in range(len(CLASSES))]
    colors     = ["#e74c3c", "#3498db", "#2ecc71", "#95a5a6"][:len(CLASSES)]
    bars       = axes[i][2].barh(CLASSES, probs_list, color=colors)
    axes[i][2].set_xlim(0, 110)
    axes[i][2].set_xlabel("Confidence (%)")
    axes[i][2].set_title("Probabilities")
    for bar, val in zip(bars, probs_list):
        axes[i][2].text(val+1, bar.get_y()+bar.get_height()/2,
                        f"{val}%", va="center", fontsize=10, fontweight="bold")
    axes[i][2].spines[["top","right"]].set_visible(False)

plt.suptitle("GradCAM — EfficientNetB2 + Spatial Attention v7",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("gradcam_v7_attention.png", dpi=150, bbox_inches="tight")
plt.show()
files.download("gradcam_v7_attention.png")

In [ ]:
!pip install grad-cam -q
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

transform_test = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

target_layer = [model.features[-1]]
from PIL import ImageOps

def auto_orient(img_pil):
    """Fix EXIF orientation and rotate landscape to portrait if needed"""
    # Fix EXIF rotation first
    img_pil = ImageOps.exif_transpose(img_pil)

    # If landscape, rotate to portrait
    w, h = img_pil.size
    if w > h:
        img_pil = img_pil.rotate(90, expand=True)

    return img_pil

def run_gradcam_on_image(img_pil, filename="image"):
    img_pil = img_pil.convert("RGB").resize((224, 224))
    img_np  = np.array(img_pil).astype(np.float32) / 255.0
    tensor  = transform_test(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, dim=1)[0]

    pred_idx   = probs.argmax().item()
    pred_brand = CLASSES[pred_idx]
    confidence = round(probs[pred_idx].item() * 100, 1)

    with GradCAM(model=model, target_layers=target_layer) as cam:
        grayscale = cam(input_tensor=tensor,
                       targets=[ClassifierOutputTarget(pred_idx)])
        cam_image = show_cam_on_image(img_np, grayscale[0], use_rgb=True)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img_np)
    axes[0].set_title(f"Original\n{filename}", fontsize=10)
    axes[0].axis("off")

    axes[1].imshow(cam_image)
    axes[1].set_title(f"GradCAM\n{pred_brand} ({confidence}%)", fontsize=10)
    axes[1].axis("off")

    probs_list = [round(probs[j].item()*100, 1) for j in range(len(CLASSES))]
    colors     = ["#e74c3c", "#3498db", "#2ecc71", "#95a5a6"][:len(CLASSES)]
    bars       = axes[2].barh(CLASSES, probs_list, color=colors)
    axes[2].set_xlim(0, 110)
    axes[2].set_xlabel("Confidence (%)")
    axes[2].set_title("Probabilities")
    for bar, val in zip(bars, probs_list):
        axes[2].text(val+1, bar.get_y()+bar.get_height()/2,
                    f"{val}%", va="center", fontsize=10, fontweight="bold")
    axes[2].spines[["top","right"]].set_visible(False)

    plt.tight_layout()
    plt.show()
    print(f"→ {pred_brand} ({confidence}%)\n")

# ── Drag and drop uploader ─────────────────────────────────────────────────
print("📂 Drag and drop images below — GradCAM runs automatically on each upload\n")

uploader = widgets.FileUpload(
    accept=".jpg,.jpeg,.png,.webp",
    multiple=True,
    description="Drop images here"
)

def on_upload(change):
    def on_upload(change):
      clear_output(wait=True)
      display(uploader)
      print(f"Processing {len(uploader.value)} image(s)...\n")
      for fname, fdata in uploader.value.items():
          img_bytes = fdata["content"]
          img_pil   = Image.open(io.BytesIO(img_bytes))
          img_pil   = auto_orient(img_pil)  # ← add this line
          run_gradcam_on_image(img_pil, filename=fname)

uploader.observe(on_upload, names="value")
display(uploader)

## Output Files

| File | Saved to | Contents / purpose |
|---|---|---|
| `classifier_best_v7_attention_3_class.pth` | `/content/` (working dir), copied to Drive `Classifier/v7_attention/`, and downloaded locally | Best-so-far checkpoint dict, overwritten each time val accuracy improves — `model_state_dict`, `best_val_acc`, `classes`, `architecture`, `num_classes`, `epochs_trained`, `train_history`, `class_counts`, `confidence_threshold`. This is the model used for evaluation/GradCAM later in the notebook. |
| `training_history_v7_attention_3_class.json` | `/content/` (working dir) | Per-epoch train/val loss and accuracy, written once at the end of training. |
| `confusion_matrix_v7_attention.png` | `/content/`, copied to Drive `Classifier/v7_attention/`, and downloaded | Raw-count and row-normalized confusion matrices on the validation set. |
| `training_history_v7_attention_3_class.png` | `/content/`, copied to Drive `Classifier/v7_attention/` as `training_history_v6_attention.png` (name changes on copy — not a typo fix, left as-is), and downloaded | Accuracy/loss curves vs. epoch. |
| `gradcam_v7_attention.png` | `/content/` and downloaded (not copied to Drive) | GradCAM overlay + per-class confidence bars for the batch of test images uploaded in the "Test images" section. |
| Ad-hoc GradCAM figures (drag-and-drop cell) | Displayed inline only, not saved to disk | Same GradCAM visualization, run per-image as files are dropped onto the `FileUpload` widget. |